# PA4 data setup: public data + supplied prefix banks (Colab T4)
Thin launcher. Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Uses the committed revision lock `splits/data_lock/local_source_revisions.json`, so the data matches what we report.
Prefix banks go through `kaggle/supplied_banks.py` (decision G4: the released script passes `generator=` that transformers 4.57.1 rejects).

## 1. GPU check and Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone repos, install pinned packages, place the revision lock

In [ ]:
%%bash
set -e
cd /content
rm -rf atml-pa4 atml-pa4-datasets
git clone -q https://github.com/Zeeba1705/atml-pa4.git
git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git
pip install -q -r atml-pa4-datasets/requirements-public.txt
mkdir -p atml-pa4-datasets/data
cp atml-pa4/splits/data_lock/local_source_revisions.json atml-pa4-datasets/data/
echo "our repo at: $(git -C atml-pa4 rev-parse --short HEAD)"
python -c "import torch, transformers, huggingface_hub; print(torch.__version__, transformers.__version__, huggingface_hub.__version__, torch.cuda.is_available())"

## 3. Public data (CPU part, a couple of minutes)

In [ ]:
%%bash
set -e
cd /content/atml-pa4-datasets
python scripts/setup_data.py --public
python scripts/check_setup.py

## 4. Quick check: banks for 2 prompts (deleted afterwards)

In [ ]:
%%bash
set -e
cd /content
python atml-pa4/kaggle/supplied_banks.py --data_root atml-pa4-datasets --limit_prefix_prompts 2
rm -f atml-pa4-datasets/data/prefix_banks/prefix_banks_locked.jsonl.gz

## 5. Full prefix banks (~30 min, keep the tab open)

In [ ]:
%%bash
set -e
cd /content
python atml-pa4/kaggle/supplied_banks.py --data_root atml-pa4-datasets
python atml-pa4-datasets/scripts/check_setup.py
ls -la atml-pa4-datasets/data/prefix_banks/

## 6. Save to Drive and confirm the lock did not change

In [ ]:
%%bash
set -e
cd /content
OUT=/content/drive/MyDrive/atml-pa4-data
mkdir -p "$OUT"
cp -r atml-pa4-datasets/data/public_locked atml-pa4-datasets/data/prefix_banks "$OUT"/
cp atml-pa4-datasets/data/local_source_revisions.json "$OUT"/
diff atml-pa4-datasets/data/local_source_revisions.json atml-pa4/splits/data_lock/local_source_revisions.json && echo "LOCK UNCHANGED"
nvidia-smi --query-gpu=name --format=csv,noheader > "$OUT"/prefix_banks_gpu.txt
ls -laR "$OUT"